# DEE — Energía de modos topológicos con winding sobre T³

**Pregunta central:** ¿Las configuraciones del campo φ con winding number no nulo alrededor de los ciclos del toro tienen ecuación de estado distinta del régimen armónico (w ≈ +0.07)?

**Hipótesis a testar:** los modos topológicos con winding tienen energía elástica que depende de la longitud global del bucle del toro, no solo de distancias locales. Su escalado con el volumen podría ser distinto, dando w_winding ≠ +0.07.

**Predicción analítica simple:** para un winding lineal φ = 2π·x/L sobre un toro de lado L, la energía elástica va como E_w ∝ K·V/L² ∝ K·L. Reescalando V → (1+ε)³V se obtiene L → (1+ε)L y E_w → (1+ε)E_w. Esto da:
$$P_w = -\partial E_w/\partial V = -\frac{1}{3} \frac{E_w}{V}$$
$$w_w = P_w/(\rho_w c^2) = -\frac{1}{3 c^2}$$

En unidades naturales con c=1: **w_w teórico = -1/3** (o más negativo si la dinámica del cristal es no trivial).

**Cuatro escenarios posibles:**

| w_winding medido | Interpretación |
|---|---|
| +0.07 | Sector topológico se comporta igual que armónico (resultado nulo) |
| -1/3 | Confirma cálculo analítico, modo de "frontera cosmológica" |
| -1 | **Λ genuina** (improbable pero buscado) |
| Otro | Comportamiento no estándar a interpretar |

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_winding'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

## Funciones del cristal y del winding

In [ ]:
def grid_T3(N_target, jitter_fraction=0.10, seed=42):
    """Cristal en T3 con condiciones periódicas."""
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def distance_matrix_T3(points, L=1.0):
    """Distancias con condiciones periódicas. También retorna el vector diferencia para signos."""
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1), diff

def build_weights(D_mat, k_neighbors=30, alpha=2.0):
    """Construye matriz de pesos w_ij = 1/d_ij^alpha (sin restar diagonal)."""
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    return W

print('Funciones listas')

## Construcción de configuraciones con winding

Un winding linear de φ alrededor del eje x significa: φ_i = 2π · x_i / L. Cuando un nodo viaja de x=0 a x=L, φ crece de 0 a 2π. Como φ es modular (definido en S¹), esto es topológicamente no trivial — equivale a dar **una vuelta** al campo escalar al recorrer una vuelta del toro.

**Energía elástica del winding:**
$$E_w = \frac{1}{2} \sum_{ij} w_{ij} (\phi_i - \phi_j)^2_{\text{periódico}}$$

donde la diferencia se calcula módulo 2π (tomando la rama más corta).

In [ ]:
def winding_field(points, axis, L_box=1.0):
    """Genera φ_i = 2π · (x_i / L_box) sobre el eje especificado (0=x, 1=y, 2=z)."""
    return 2 * np.pi * points[:, axis] / L_box

def winding_energy(W, points, axis, L_box=1.0):
    """
    Calcula E = ½ Σ w_ij (φ_i - φ_j)²_periódica
    
    Para un winding lineal φ = 2π·x/L, la diferencia entre nodos vecinos debe
    contemplar la geometría periódica del toro. Cuando dos nodos están conectados
    *a través del borde periódico*, ambos están del lado opuesto del corte: la
    diferencia natural en φ debe ser proporcional a la diferencia GEOMÉTRICA corta
    en la coordenada, no a la diferencia naive (x_i - x_j) que sería grande.
    
    Implementación: para cada par (i,j) con peso w_ij,
        Δx_geom = (x_i - x_j) - L · round((x_i - x_j) / L)
        Δφ = 2π · Δx_geom / L
    Esto garantiza ∇φ ≈ 2π/L uniforme sobre todo el toro.
    """
    W_coo = W.tocoo()
    rows = W_coo.row
    cols = W_coo.col
    vals = W_coo.data
    
    # Diferencia GEOMÉTRICA periódica en el eje
    dx_geom = points[rows, axis] - points[cols, axis]
    dx_geom = dx_geom - L_box * np.round(dx_geom / L_box)
    
    # Diferencia en φ correspondiente al winding lineal
    dphi = 2 * np.pi * dx_geom / L_box
    
    # Energía: factor ½ del Hamiltoniano elástico ½ Σ w_ij (φ_i - φ_j)²
    # W simetrizada cuenta cada par dos veces (ij y ji), por eso dividimos por 2 extra
    E = 0.5 * 0.5 * np.sum(vals * dphi**2)
    
    return E

print('Funciones de winding listas')

## Test 1 — Energía del winding a N fijo

Calculamos E_winding sobre los tres ejes (x, y, z). Por isotropía esperamos los tres valores casi iguales.

In [ ]:
N_TEST = 1331
K_NEIGHBORS = 30
JITTER = 0.10

print(f'Test 1: energía del winding con N = {N_TEST}, kernel 1/d²\n')

points, N = grid_T3(N_TEST, JITTER, seed=42)
D, _ = distance_matrix_T3(points, L=1.0)
W = build_weights(D, K_NEIGHBORS, alpha=2.0)

# Calcular energía para winding en cada eje
energies = {}
for axis_name, axis in zip(['x', 'y', 'z'], [0, 1, 2]):
    E = winding_energy(W, points, axis, L_box=1.0)
    energies[axis_name] = E
    print(f'  E_winding ({axis_name}) = {E:.4f}')

E_w_avg = np.mean(list(energies.values()))
E_w_std = np.std(list(energies.values()))
print(f'\n  Promedio: {E_w_avg:.4f} ± {E_w_std:.4f}')
print(f'  Anisotropía relativa: {E_w_std/E_w_avg*100:.2f}%')

if E_w_std/E_w_avg < 0.05:
    print(f'  → ✓ Energía isotrópica (consistente con cristal isotrópico)')
else:
    print(f'  → Anisotropía detectada')

## Test 2 — Escalamiento de E_winding con el volumen

Variamos el tamaño de la caja (5 valores de ε) manteniendo N fijo. Si el winding tiene presión negativa, el escalamiento de E_w(V) será distinto al de la energía armónica.

In [ ]:
epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]

print(f'Test 2: escalamiento E_winding(V) con N fijo = {N_TEST}\n')
print(f'{"ε":>6} {"L":>8} {"V":>8} {"E_w_x":>10} {"E_w_y":>10} {"E_w_z":>10} {"<E_w>":>10}')
print('-'*70)

data = []
for eps in epsilons:
    L_box = 1.0 + eps
    V_box = L_box**3
    
    # Reescalear puntos con la caja
    points_eps = points * L_box
    D_eps, _ = distance_matrix_T3(points_eps, L=L_box)
    W_eps = build_weights(D_eps, K_NEIGHBORS, alpha=2.0)
    
    # Calcular energía de winding en cada eje
    Es = []
    for axis in [0, 1, 2]:
        E = winding_energy(W_eps, points_eps, axis, L_box=L_box)
        Es.append(E)
    
    E_avg = np.mean(Es)
    
    data.append({
        'eps': eps, 'L': L_box, 'V': V_box,
        'E_x': Es[0], 'E_y': Es[1], 'E_z': Es[2],
        'E_avg': E_avg
    })
    
    print(f'{eps:+.3f} {L_box:>8.4f} {V_box:>8.4f} '
          f'{Es[0]:>10.4f} {Es[1]:>10.4f} {Es[2]:>10.4f} {E_avg:>10.4f}')

In [ ]:
# Calcular presión y w del winding
Vs = np.array([d['V'] for d in data])
Es = np.array([d['E_avg'] for d in data])

idx_0 = list(epsilons).index(0.0)

# Diferencia finita centrada
dE_dV = (Es[idx_0+1] - Es[idx_0-1]) / (Vs[idx_0+1] - Vs[idx_0-1])
P_w = -dE_dV
rho_w = Es[idx_0] / Vs[idx_0]

# Velocidad del sonido del cristal (de SIM 11)
c_s = 2.186

# w del winding
w_winding = P_w / (rho_w * c_s**2)

# También calculo w respecto a c=1 (unidades naturales)
w_winding_c1 = P_w / rho_w

# Ajuste de ley de potencia: E_w ∝ V^β
log_V = np.log10(Vs)
log_E = np.log10(Es)
beta_w, log_A = np.polyfit(log_V, log_E, 1)

print('='*70)
print('RESULTADO: ecuación de estado del winding topológico')
print('='*70)
print()
print(f'Ajuste E_w(V) ∝ V^β:')
print(f'  β_winding = {beta_w:+.4f}')
print(f'  Predicción analítica para winding lineal en cristal isotrópico: β = +1/3 ≈ +0.333')
print(f'  Predicción para sector armónico: β = -1/3 ≈ -0.333 (E_arm decrece con V)')
print()
print(f'Cálculo de presión y w:')
print(f'  E_w(V_central) = {Es[idx_0]:.4f}')
print(f'  ∂E_w/∂V        = {dE_dV:+.4f}')
print(f'  P_w            = {P_w:+.4f}')
print(f'  ρ_w            = {rho_w:+.4f}')
print(f'  w_w (con c_s²) = {w_winding:+.4f}')
print(f'  w_w (con c=1)  = {w_winding_c1:+.4f}')
print()
print(f'Comparación con benchmarks:')
print(f'  w cristal armónico  (SIM 16) = +0.0697')
print(f'  w defecto blando    (SIM 17) = +0.0698')
print(f'  w diferencia topol. (notebook anterior) = +0.0698')
print(f'  w winding topológ.  (este test, c=1)    = {w_winding_c1:+.4f}')
print(f'  w para Λ                              = -1.0000')
print(f'  w para frontera cosmológica           = -1/3 = -0.3333')

In [ ]:
# Veredicto físico (usando w con c=1 que es el natural para este problema)
w_use = w_winding_c1

print('='*70)
print('VEREDICTO FÍSICO')
print('='*70)
print()

if abs(w_use - 0.07) < 0.05:
    print(f'  w_winding = {w_use:+.3f} ≈ +0.07')
    print(f'  → IDÉNTICO al sector armónico')
    print(f'  → El winding NO da física distinta al armónico estándar')
    print(f'  → CONFIRMA que w ≈ +0.07 es propiedad estructural del kernel 1/d²')
    print(f'    para CUALQUIER configuración (vibracional, defecto, topológica)')
    print()
    print(f'  IMPLICACIÓN: el cristal armónico DEE no produce Λ por ningún')
    print(f'  mecanismo dentro de su régimen actual. Λ requiere salir del')
    print(f'  régimen armónico (anharmonicidad, transiciones de fase, etc.)')
    print(f'  o un mecanismo no contemplado en el modelo actual.')
elif abs(w_use + 1/3) < 0.1:
    print(f'  w_winding = {w_use:+.3f} ≈ -1/3')
    print(f'  → Coincide con predicción analítica de winding lineal')
    print(f'  → DISTINTO del sector armónico')
    print(f'  → Confirma física topológica genuina, pero NO es Λ pura')
    print(f'  → Análogo a "frontera cosmológica": presión justo en el límite')
    print(f'    entre desaceleración y aceleración')
    print()
    print(f'  IMPLICACIÓN: nueva clase de fenomenología en DEE')
    print(f'  Podría contribuir parcialmente a la aceleración cósmica')
elif abs(w_use + 1) < 0.15:
    print(f'  w_winding = {w_use:+.3f} ≈ -1')
    print(f'  → ✓✓✓ COMPATIBLE CON CONSTANTE COSMOLÓGICA')
    print(f'  → El sector topológico de winding produce Λ verdadera')
    print(f'  → RESULTADO MUY FUERTE para DEE')
    print()
    print(f'  IMPLICACIÓN: la energía oscura emerge de configuraciones')
    print(f'  topológicamente no triviales del cristal sobre T³.')
    print(f'  Esto requeriría verificación adicional independiente.')
elif w_use < -0.5 and w_use > -1:
    print(f'  w_winding = {w_use:+.3f}')
    print(f'  → Presión negativa fuerte pero no exactamente -1')
    print(f'  → Podría ser energía oscura dinámica (quintaesencia)')
    print(f'  → Diferente del sector armónico, físicamente novedoso')
elif w_use > 0.2:
    print(f'  w_winding = {w_use:+.3f}')
    print(f'  → Presión positiva grande, comportamiento tipo radiación/materia')
    print(f'  → No contribuye a aceleración cósmica')
else:
    print(f'  w_winding = {w_use:+.3f}')
    print(f'  → Comportamiento intermedio, requiere análisis adicional')

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: E_w vs V (escala lineal y ajuste)
ax = axes[0]
ax.plot(Vs, Es, 'o-', markersize=14, lw=2, color='darkblue')
# Ajuste
V_smooth = np.linspace(Vs.min(), Vs.max(), 50)
E_fit = 10**log_A * V_smooth**beta_w
ax.plot(V_smooth, E_fit, '--', color='red', lw=2, alpha=0.7, label=f'Ajuste β={beta_w:+.3f}')
ax.set_xlabel('V (volumen)', fontsize=12)
ax.set_ylabel('E_winding (energía elástica)', fontsize=12)
ax.set_title('E_winding(V): escalamiento del sector topológico', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Panel 2: comparación de w con benchmarks
ax = axes[1]
categories = ['Λ\n(w=-1)', 'Frontera\n(w=-1/3)', 'Polvo\n(w=0)', 
              'Cristal\n(w=+0.07)', 'Defecto\n(w=+0.07)',
              'Topología\n(w=+0.07)', f'WINDING\n(w={w_use:+.3f})']
values = [-1, -1/3, 0, 0.07, 0.07, 0.07, w_use]
colors = ['green', 'purple', 'gray', 'steelblue', 'orange', 'red', 'crimson']
ax.barh(categories, values, color=colors, alpha=0.7)
ax.axvline(0, color='black', lw=1)
ax.axvline(-1, color='green', linestyle='--', alpha=0.5, label='Λ requerido')
ax.axvline(-1/3, color='purple', linestyle='--', alpha=0.5, label='Frontera aceleración')
ax.set_xlabel('Parámetro de ecuación de estado w', fontsize=12)
ax.set_title('w_winding vs benchmarks cosmológicos', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
save_plot('43_w_winding')
plt.show()

In [ ]:
# Síntesis final
print('='*75)
print('SÍNTESIS — Modos topológicos con winding en T³')
print('='*75)
print()
print(f'Configuración testada: φ_i = 2π · x_i / L (winding 1 en cada eje)')
print(f'Cristal: N = {N_TEST}, kernel 1/d², jitter = {JITTER}')
print()
print(f'Energía del winding (eje x, N=1331): E_w = {data[idx_0]["E_x"]:.4f}')
print(f'Anisotropía entre ejes: {E_w_std/E_w_avg*100:.2f}%')
print()
print(f'Escalamiento E_w(V): β = {beta_w:+.3f}')
print(f'Presión topológica: P_w = {P_w:+.4f}')
print(f'Densidad: ρ_w = {rho_w:.4f}')
print(f'w del winding (c=1): {w_winding_c1:+.4f}')
print()

if abs(w_winding_c1 - 0.07) < 0.05:
    print('CONCLUSIÓN: el sector topológico de winding sobre T³ tiene la misma')
    print('ecuación de estado que el sector armónico (w ≈ +0.07). Esto cierra')
    print('el ciclo: w ≈ +0.07 es propiedad estructural del kernel 1/d² del')
    print('sustrato, INDEPENDIENTE de la configuración (vibracional, defecto,')
    print('o topológica). El cristal armónico DEE no produce Λ por ningún')
    print('mecanismo dentro de su régimen actual.')
    print()
    print('Esto NO es una falla del modelo: es una caracterización honesta')
    print('de su rango de validez. La pregunta sobre Λ permanece abierta y')
    print('requiere ingredientes fuera del régimen armónico (anharmonicidad,')
    print('transiciones de fase del kernel, conexión con escala QCD, etc.)')
elif abs(w_winding_c1 + 1/3) < 0.1:
    print('CONCLUSIÓN: el sector topológico de winding tiene w ≈ -1/3,')
    print('coincidente con la predicción analítica de winding lineal y')
    print('DIFERENTE del sector armónico (+0.07). Esto abre una nueva')
    print('dirección física en DEE: configuraciones topológicas tienen')
    print('fenomenología cosmológica distinta del régimen vibracional.')
elif abs(w_winding_c1 + 1) < 0.15:
    print('CONCLUSIÓN: w ≈ -1 — RESULTADO FUERTE.')
    print('El sector topológico de winding sobre T³ es candidato genuino')
    print('a constante cosmológica. Requiere verificación adicional con')
    print('métodos independientes antes de declarar resultado definitivo.')
else:
    print(f'CONCLUSIÓN: w_winding = {w_winding_c1:+.3f} no coincide con')
    print('ningún benchmark estándar. Requiere análisis adicional.')

In [ ]:
import shutil
shutil.make_archive('plots_winding', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_winding.zip')

try:
    from google.colab import files
    files.download('plots_winding.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass